# 01 — SQL Foundations & the Relational Model

**What's in this notebook:**
- What SQL is, and why a 50-year-old language still runs the data world
- The relational model — tables, rows, columns, and types
- Keys and constraints — how the database enforces correctness
- Normalization — what 1NF / 2NF / 3NF actually mean in practice
- Declarative thinking — you say *what*, the engine decides *how*
- The SQL dialect landscape — Postgres, MySQL, SQL Server, Snowflake, DuckDB
- Hands-on: spin up DuckDB inside this notebook and create the first schema

## 1. What SQL is

SQL stands for **Structured Query Language**. It's a *declarative* language for talking to relational databases: you describe the result you want, the engine figures out how to produce it.

Born at IBM in the 1970s, standardized as ANSI SQL, and still the lingua franca of data tooling today — OLTP databases, data warehouses, embedded engines, BI tools, and stream processors all speak some flavor of it.

If you're coming from the JVM, think of Java Streams — `list.stream().filter(...).map(...).collect(...)` is declarative in the same spirit. SQL takes that idea further: the engine can reorder operations, pick join algorithms, use indexes, and parallelize work — all from the same query text.

## 2. The relational model

Three pieces of vocabulary do almost all the work:

- **Relation** — a table
- **Tuple** — a row
- **Attribute** — a column

A table has a fixed set of typed columns. Every row is one fact stated in those columns. **Row order is not guaranteed** — if you want a specific order in the result, you ask for it with `ORDER BY`.

```
  customers
  +-------------+--------+---------------------+
  | customer_id | name   | email               |
  +-------------+--------+---------------------+
  |     1       | Alice  | alice@example.com   |
  |     2       | Bob    | bob@example.com     |
  |     3       | Carol  | NULL                |
  +-------------+--------+---------------------+
       ^             ^             ^
    column        column        column
  (each row is one tuple/fact)
```

### Hands-on: connect to DuckDB

DuckDB is an embedded analytical database — no server, just a file. We'll load the `sql` magic so we can run queries directly in cells.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

In [ ]:
%%sql
DROP TABLE IF EXISTS customers;
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    name          VARCHAR  NOT NULL,
    email         VARCHAR  UNIQUE,
    signup_date   DATE     NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Alice', 'alice@example.com', DATE '2024-01-15'),
    (2, 'Bob',   'bob@example.com',   DATE '2024-02-03'),
    (3, 'Carol', NULL,                DATE '2024-03-20');
SELECT * FROM customers;

## 3. Keys and constraints

Constraints are how the database refuses to store invalid data. The engine enforces them on every insert and update — bad data never lands.

- **Primary key** — uniquely identifies a row. Equivalent to `NOT NULL` + `UNIQUE`, plus it's the table's identity column.
- **Foreign key** — a column that must match a primary key in another table. The engine refuses inserts that don't resolve, and refuses deletes that would orphan rows.
- **UNIQUE** — no two rows share this value (NULLs allowed, and usually treated as distinct).
- **NOT NULL** — a value is required.
- **CHECK** — an arbitrary boolean condition each row must satisfy.

Foreign keys are how the relational model glues independent facts together: each `orders` row points at exactly one `customers` row.

In [ ]:
%%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL REFERENCES customers(customer_id),
    amount       DECIMAL(10,2) CHECK (amount > 0),
    order_date   DATE NOT NULL
);
INSERT INTO orders VALUES
    (101, 1, 29.99,  DATE '2024-04-01'),
    (102, 1, 149.50, DATE '2024-04-12'),
    (103, 2, 9.99,   DATE '2024-04-15');
SELECT * FROM orders;

In [ ]:
%%sql
-- Constraint refuses bad data: customer_id 99 does not exist.
-- This insert will fail with a foreign key violation.
INSERT INTO orders VALUES (999, 99, 10.00, DATE '2024-05-01');

## 4. Normalization in one page

Normalization is the rulebook for *how to split data into tables*. There are higher forms, but in practice three cover almost everything:

- **1NF** — every cell holds one atomic value. No comma-separated lists, no JSON blobs standing in for what should be rows.
- **2NF** — non-key columns depend on the **whole** primary key. Only matters when the key is composite.
- **3NF** — non-key columns depend on **nothing but** the primary key. No transitive dependencies like `order → customer_id → customer_address`.

The intuition: **things that vary independently belong in their own table, connected by foreign keys.** A customer's address that changes shouldn't force you to update every order row they ever placed.

```
  Not normalized:                  Normalized (3NF):
  +----+-------+----------+        +----+-------+        +----+----+--------+
  | id | cust  | address  |        | id | cust  |        | id | cid| addr   |
  +----+-------+----------+        +----+-------+        +----+----+--------+
  | 1  | Alice | 12 Main  |        | 1  | Alice |        | 1  | 1  | 12 Main|
  | 2  | Alice | 12 Main  |        | 2  | Bob   |        | 2  | 2  | 5 Oak  |
  | 3  | Bob   | 5 Oak    |        +----+-------+        +----+----+--------+
  +----+-------+----------+        customers              addresses
    (address duplicated)             (one fact            (separate, joinable)
                                      per row)
```

## 5. Declarative thinking

The hardest mental shift coming from imperative code is letting go of *how*. You don't write loops. You don't pick index lookups. You write a question:

> *"Give me each customer's total spend, highest first."*

The engine then plans it — which index to scan, which join algorithm to pick, whether to parallelize across cores. That plan can change as the data grows or as you add indexes, without rewriting the query.

**Rule of thumb:** trust the planner first. Only override (with hints, materialized CTEs, or rewrites) when `EXPLAIN` shows it's actually wrong. We'll meet `EXPLAIN` properly in notebook 08.

In [ ]:
%%sql
-- One question, no loops. The engine handles the rest.
SELECT c.name,
       COUNT(o.order_id) AS order_count,
       COALESCE(SUM(o.amount), 0) AS total_spent
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.name
ORDER BY total_spent DESC;

## 6. The SQL dialect landscape

Most everyday SQL is portable. The places dialects diverge: window function syntax, date and time functions, JSON and array handling, and `MERGE` / upsert syntax.

- **PostgreSQL** — open source, the closest thing to a reference dialect, rich feature set.
- **MySQL / MariaDB** — web-stack default, historically looser semantics, catching up in MySQL 8+.
- **SQL Server (T-SQL)** — Microsoft enterprise, `TOP` instead of `LIMIT`.
- **SQLite** — embedded, single file, lives inside every phone and browser.
- **DuckDB** — embedded analytics, columnar storage — what we're using here.
- **Snowflake / BigQuery / Redshift** — cloud warehouses, each with their own quirks (notably window-function extensions like `QUALIFY`).

We'll write **standard ANSI SQL** throughout this series and call out dialect-specific syntax explicitly when it comes up.

## 7. Common pitfalls (carry these forward)

- **Forgetting `WHERE` on `UPDATE` or `DELETE`** — touches every row in the table. Run as `SELECT` first to preview.
- **Comparing to NULL with `=`** — NULL is not equal to anything, including itself. `x = NULL` is always unknown (treated as false). Use `x IS NULL` / `x IS NOT NULL`.
- **`SELECT *` in production code** — when columns change, downstream code breaks silently. Name the columns you need.
- **Missing primary key** — without one, the engine can't enforce row identity, and many ORMs, replication tools, and BI tools misbehave.
- **Trusting row order without `ORDER BY`** — the engine is free to return rows in any order it likes. Today's order is not tomorrow's guarantee.

Next up: **notebook 02 — Querying with SELECT**, where we use these tables to learn the read path properly.